<a href="https://colab.research.google.com/github/harinimanohar06/AI_model/blob/main/genAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 24.4 MB/s eta 0:00:00
--2026-04-17 10:05:54--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-17 10:05:54--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-17T10%3A55%3A48Z&rscd=attachment%3B+filename%3Dcl

In [2]:
pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00


In [3]:
%%writefile app.py
import streamlit as st
from transformers import pipeline
from groq import Groq
import os
from huggingface_hub import InferenceClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import os
from google.colab import userdata
HF_TOKEN=userdata.get('HF_Token')
GROQ_API_KEY=userdata.get('GROQ_API_KEY')
# ---------------------------
# LOAD MODELS
# ---------------------------
@st.cache_resource
def load_translation_model():
    model_name = "Helsinki-NLP/opus-mt-mul-en"

    # Load model and tokenizer manually
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Create a text2text pipeline manually
    return pipeline(
        task="text-generation",  # only available in v5
        model=model,
        tokenizer=tokenizer,
        device=-1,  # CPU
    )
@st.cache_resource
def load_hf_client():
    from huggingface_hub import InferenceClient
    return InferenceClient(api_key=HF_TOKEN)


translator = load_translation_model()
client_groq = Groq(api_key=GROQ_API_KEY)
hf_client = load_hf_client()
# ---------------------------
# FUNCTIONS
# ---------------------------
def translate_text(tamil_text):
    return translator(tamil_text)[0]['generated_text']
def generate_text(prompt):
    response = client_groq.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a creative writer."},
            {"role": "user", "content": f"Convert this into a detailed image prompt: {prompt}"}
        ]
    )
    return response.choices[0].message.content
def generate_image(prompt):
    image = hf_client.text_to_image(
        prompt,
        model="stabilityai/stable-diffusion-xl-base-1.0",
    )
    return image

# ---------------------------
# UI
# ---------------------------
st.title("🎨 Tamil → Image Generator")

tamil_input = st.text_area("Enter Tamil Text")

if st.button("Generate"):
    if tamil_input.strip() == "":
        st.warning("Please enter text")
    else:
        with st.spinner("Processing..."):
            # Step 1: Translate
            translated = translate_text(tamil_input)
            st.subheader("Translated Text")
            st.write(translated)
            # Step 2: Generate Prompt
            generated_text = generate_text(translated)
            st.subheader("Generated Prompt")
            st.write(generated_text)

            # Step 3: Generate Image
            image = generate_image(generated_text)
            st.subheader("Generated Image")
            st.image(image, use_column_width=True)
            st.success("Done!")



Writing app.py


In [4]:
!streamlit run /content/app.py &>/content/logs.txt &  # here instead of app.py please rename with your file name

In [5]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://hierarchy-lonely-qty-relay.trycloudflare.com
